# Modelo de reglas con scores por clase

Notebook organizado por bloques para Google Colab. Ejecuta las celdas en orden.


In [42]:

from __future__ import annotations

import re
import sys
import unicodedata
from datetime import datetime
from pathlib import Path
from typing import Iterable

import medspacy
import pandas as pd
from spacy.language import Language
from spacy.tokens import Span


## 2. Configuracion Principal


In [43]:

EXCEL_PATH = Path(
    r"C:\Users\mafed\Desktop\Trabajo de grado\Datos\datos limpios\base etiquetada limpia.xlsx"
)

ID_COLUMN = "ID_PSEUDO"
LABEL_COLUMN = "Mención de cáncer"
TRAIN_FRACTION = 0.8
VALIDATION_WITHIN_TRAIN_FRACTION = 0.2
RANDOM_STATE = 42

RESULTS_DIR = Path(
    r"C:\Users\mafed\Desktop\Trabajo de grado\Datos\Resultados\Reglas"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RESULTS_DIR / f"{EXCEL_PATH.stem}_clasificada_reglas_quickumls_con_scores.xlsx"

POSITIVE_CLASSES = (1, 2)
PRIMARY_REVIEW_CLASSES = (1, 2)
LEFT_CONTEXT_WINDOW = 80
SURROUNDING_CONTEXT_WINDOW = 40
MIN_POSITIVE_RATE_TO_KEEP_POSITIVE = 0.0
CLASS2_RECALL_BOOST = 2.40
CLASS1_DEFINITE_BOOST = 1.05
CLASS2_SELECTION_RATIO = 0.80
CLASS2_DIRECT_EVIDENCE_MIN = 2
MAX_ERROR_EXAMPLES = 10
VERBOSE = False

SCORE_MARGIN_LOW = 0.0
SCORE_MARGIN_MEDIUM = 1.0
SCORE_MARGIN_HIGH = 2.5

if not EXCEL_PATH.exists():
    raise FileNotFoundError(
        f"No se encontro el archivo en la ruta indicada:\n{EXCEL_PATH}"
    )


def vprint(*args, **kwargs):
    if VERBOSE:
        print(*args, **kwargs)


## 3. Diccionarios De Reglas


In [44]:

CANCER_SEMTYPES = {"T191"}

KEYWORD_TERMS = {
    "cancer", "canceres", "carcinoma", "carcinomatosis", "adenocarcinoma",
    "tumor", "tumores", "tumoracion", "neoplasia", "neoplasias",
    "neoplasico", "neoplastico", "maligno", "maligna", "malignos", "malignas",
    "malignidad", "metastasis", "metastatico", "metastatica", "metastasico",
    "metastasica", "oncologia", "oncologico", "oncologica",
    "leucemia", "leucemias", "linfoma", "linfomas", "sarcoma", "sarcomas",
    "melanoma", "melanomas", "mieloma", "mielomas", "glioma", "glioblastoma",
    "blastoma", "blastomas", "neuroblastoma", "retinoblastoma",
    "hepatocarcinoma", "colangiocarcinoma", "mesotelioma", "seminoma",
    "paraganglioma", "timoma", "germinoma", "mielodisplasico",
    "mieloproliferativa", "mieloproliferativo", "mielofibrosis",
    "policitemia", "gist", "urotelial", "carcinosis"
}

KEYWORD_PHRASES = {
    "carcinoma in situ",
    "neoplasia maligna",
    "tumor maligno",
    "tumores malignos",
    "lesion maligna",
    "lesiones malignas",
    "masa tumoral",
    "proceso neoplasico",
    "enfermedad neoplasica",
    "neoplasia metastasica",
    "enfermedad metastasica",
    "sospecha de cancer",
    "diagnostico de cancer",
    "dx de cancer",
    "cancer de",
    "cancer del",
    "cancer en",
    "linfoma de hodgkin",
    "linfoma no hodgkin",
    "mieloma multiple",
    "leucemia linfocitica",
    "leucemia mieloide",
    "carcinoma ductal",
    "carcinoma lobulillar",
    "carcinoma escamoso",
    "carcinoma basocelular",
    "carcinoma epidermoide",
    "neoplasia intraepitelial",
    "neoplasia mieloproliferativa",
    "enfermedad mieloproliferativa",
    "tumor neuroendocrino",
    "tumor de ewing",
    "tumor de wilms",
    "sindrome mielodisplasico",
    "policitemia vera",
    "sospecha de tumoracion maligna",
    "sospecha de malignidad",
    "comportamiento incierto"
}

DEFINITIVE_PHRASES = {
    "diagnostico de cancer",
    "dx de cancer",
    "neoplasia maligna",
    "tumor maligno",
    "tumores malignos",
    "lesion maligna",
    "lesiones malignas",
    "neoplasia metastasica",
    "enfermedad metastasica",
    "carcinoma in situ",
}

DEFINITIVE_EXACT_TERMS = {
    "cancer",
    "canceres",
    "metastasis",
    "metastatico",
    "metastatica",
    "metastasico",
    "metastasica",
    "maligno",
    "maligna",
    "leucemia",
    "linfoma",
    "mieloma",
}

DEFINITIVE_SUFFIXES = (
    "carcinoma",
    "sarcoma",
    "melanoma",
    "mieloma",
    "glioblastoma",
    "blastoma",
    "leucemia",
    "linfoma",
)

NEGATION_CUES = {
    "sin evidencia de",
    "no evidencia de",
    "niega",
    "descarta",
    "descartado",
    "negativo para",
    "sin signos de",
    "no se observa",
    "ausencia de",
}

UNCERTAINTY_CUES = {
    "sospecha de",
    "probable",
    "probablemente",
    "posible",
    "en estudio",
    "a descartar",
    "compatible con",
    "sugestivo de",
    "comportamiento incierto",
    "vs ",
    "sugiere",
    "sugerente de",
    "por descartar",
    "no se descarta",
    "presuntivo",
    "presuntiva",
    "pendiente de confirmacion",
    "pendiente de estudio",
    "sospechoso de",
    "sospechosa de",
    "alta sospecha de",
    "podria corresponder a",
    "puede corresponder a",
    "a considerar",
    "no es posible descartar",
    "sin descartar",
    "hallazgo sugestivo de",
    "hallazgos sugestivos de",
    "lesion sospechosa",
    "masa sospechosa",
    "aparente neoplasia",
    "aparente malignidad",
}

FAMILY_CUES = {
    "antecedente familiar de",
    "familiar con",
    "familiares con",
    "madre con",
    "padre con",
    "hermano con",
    "hermana con",
}

HISTORY_CUES = {
    "antecedente de",
    "antecedentes de",
    "historia de",
    "status post",
    "post tratamiento de",
}

KEYWORD_PHRASE_TOKENS = [phrase.split() for phrase in sorted(KEYWORD_PHRASES)]


## 4. Funciones Auxiliares


In [45]:

def normalize_for_match(text: str) -> str:
    text = "" if text is None else str(text)
    normalized = unicodedata.normalize("NFKD", text)
    normalized = "".join(ch for ch in normalized if not unicodedata.combining(ch))
    return normalized.lower()



def row_to_text(values: Iterable[str]) -> str:
    cleaned = []
    for value in values:
        value = str(value).strip()
        if value and value != "0" and value.lower() != "no presento ninguno":
            cleaned.append(value)
    return " ".join(cleaned).strip()



def is_definitive_cancer_term(key: str) -> bool:
    if key in DEFINITIVE_PHRASES or key in DEFINITIVE_EXACT_TERMS:
        return True
    if key.startswith("cancer "):
        return True
    key_singular = key[:-1] if key.endswith("s") else key
    return any(key_singular.endswith(suffix) for suffix in DEFINITIVE_SUFFIXES)



def get_local_window(
    text: str,
    start: int,
    end: int,
    left_window: int = SURROUNDING_CONTEXT_WINDOW,
    right_window: int = SURROUNDING_CONTEXT_WINDOW,
) -> str:
    left = max(0, start - left_window)
    right = min(len(text), end + right_window)
    return text[left:right]



def get_left_window(
    text: str,
    start: int,
    left_window: int = LEFT_CONTEXT_WINDOW,
) -> str:
    left = max(0, start - left_window)
    return text[left:start]



def has_any_cue(text: str, cues: set[str]) -> bool:
    return any(cue in text for cue in cues)



def _has_overlap(span: Span, occupied_tokens: set[int]) -> bool:
    return any(i in occupied_tokens for i in range(span.start, span.end))



def _keyword_spans(doc) -> list[Span]:
    tokens_norm = [normalize_for_match(t.text) for t in doc]
    spans: list[Span] = []
    seen: set[tuple[int, int]] = set()

    for i, tok in enumerate(tokens_norm):
        if tok in KEYWORD_TERMS:
            key = (i, i + 1)
            if key not in seen:
                spans.append(Span(doc, i, i + 1, label="CANCER_KEYWORD"))
                seen.add(key)

    for phrase_tokens in KEYWORD_PHRASE_TOKENS:
        length = len(phrase_tokens)
        if length == 0 or length > len(tokens_norm):
            continue
        for i in range(len(tokens_norm) - length + 1):
            if tokens_norm[i:i + length] == phrase_tokens:
                key = (i, i + length)
                if key not in seen:
                    spans.append(Span(doc, i, i + length, label="CANCER_KEYWORD"))
                    seen.add(key)

    return spans



def cancer_keyword_component(doc):
    new_spans = _keyword_spans(doc)
    if not new_spans:
        return doc

    ents = list(doc.ents)
    occupied = set()
    for ent in ents:
        for i in range(ent.start, ent.end):
            occupied.add(i)

    for span in new_spans:
        if _has_overlap(span, occupied):
            continue
        ents.append(span)
        for i in range(span.start, span.end):
            occupied.add(i)

    doc.ents = ents
    return doc


if "cancer_keyword_component" not in Language.factories:
    Language.component("cancer_keyword_component", func=cancer_keyword_component)



def extract_rule_features(doc) -> dict:
    doc_text_norm = normalize_for_match(doc.text)

    valid_hits = []
    negated_hits = []

    for ent in doc.ents:
        key = normalize_for_match(ent.text)

        left_window = get_left_window(doc_text_norm, ent.start_char, LEFT_CONTEXT_WINDOW)
        surrounding_window = get_local_window(
            doc_text_norm,
            ent.start_char,
            ent.end_char,
            left_window=SURROUNDING_CONTEXT_WINDOW,
            right_window=SURROUNDING_CONTEXT_WINDOW,
        )

        negated = has_any_cue(left_window, NEGATION_CUES)
        uncertain = has_any_cue(left_window, UNCERTAINTY_CUES) or has_any_cue(
            surrounding_window, UNCERTAINTY_CUES
        )
        family = has_any_cue(left_window, FAMILY_CUES)
        historical = has_any_cue(left_window, HISTORY_CUES)

        umls_matches = []
        if ent.label_ != "CANCER_KEYWORD":
            umls_matches = getattr(ent._, "umls_matches", []) or []

        semtype_hit = False
        for match in umls_matches:
            semtypes = getattr(match, "semtypes", []) or []
            if any(st in CANCER_SEMTYPES for st in semtypes):
                semtype_hit = True
                break

        is_definitive = semtype_hit or is_definitive_cancer_term(key)

        hit_info = {
            "term": key,
            "label": ent.label_,
            "semtype_hit": semtype_hit,
            "definitive": is_definitive,
            "negated": negated,
            "uncertain": uncertain,
            "family": family,
            "historical": historical,
        }

        valid_hits.append(hit_info)
        if negated:
            negated_hits.append(hit_info)

    evidence_hits = [
        hit for hit in valid_hits
        if not hit["negated"] and not hit["family"] and not hit["historical"]
    ]

    negated_case_hits = [
        hit for hit in valid_hits
        if hit["negated"] and not hit["family"] and not hit["historical"]
    ]

    context_only_hits = [
        hit for hit in valid_hits
        if hit["family"] or hit["historical"]
    ]

    definitive_present = any(hit["definitive"] for hit in evidence_hits)

    if definitive_present:
        rule_state = "definite_positive"
    elif evidence_hits:
        rule_state = "weak_positive"
    elif negated_case_hits:
        rule_state = "negated_evidence"
    else:
        rule_state = "no_evidence"

    n_valid_hits = len(valid_hits)
    n_ignored_hits = 0
    n_negated_hits = len(negated_hits)
    n_evidence_hits = len(evidence_hits)
    n_negated_case_hits = len(negated_case_hits)
    n_context_only_hits = len(context_only_hits)
    n_definitive_hits = int(sum(hit["definitive"] for hit in evidence_hits))
    n_uncertain_hits = int(sum(hit["uncertain"] for hit in evidence_hits))
    n_family_hits = int(sum(hit["family"] for hit in valid_hits))
    n_historical_hits = int(sum(hit["historical"] for hit in valid_hits))

    assign_direct_class_0 = n_evidence_hits == 0 and n_negated_case_hits > 0

    uncertainty_dominates = n_uncertain_hits >= max(1, n_definitive_hits)
    assign_direct_class_2 = (
        n_evidence_hits > 0 and (
            (n_uncertain_hits > 0 and n_definitive_hits == 0)
            or (
                n_evidence_hits >= CLASS2_DIRECT_EVIDENCE_MIN
                and n_uncertain_hits > 0
                and uncertainty_dominates
            )
        )
    )

    detected_terms = sorted({hit["term"] for hit in evidence_hits})
    context_terms = sorted({hit["term"] for hit in context_only_hits})

    return {
        "rule_state": rule_state,
        "assign_direct_class_0": bool(assign_direct_class_0),
        "assign_direct_class_2": bool(assign_direct_class_2),
        "detected_terms": detected_terms,
        "context_terms": context_terms,
        "n_valid_hits": int(n_valid_hits),
        "n_ignored_hits": int(n_ignored_hits),
        "n_negated_hits": int(n_negated_hits),
        "n_evidence_hits": int(n_evidence_hits),
        "n_negated_case_hits": int(n_negated_case_hits),
        "n_context_only_hits": int(n_context_only_hits),
        "n_definitive_hits": int(n_definitive_hits),
        "n_uncertain_hits": int(n_uncertain_hits),
        "n_family_hits": int(n_family_hits),
        "n_historical_hits": int(n_historical_hits),
    }



def build_decision_profile(
    n_evidence_hits: int,
    n_definitive_hits: int,
    n_uncertain_hits: int,
    n_negated_case_hits: int = 0,
) -> str:
    if int(n_evidence_hits) <= 0:
        if int(n_negated_case_hits) > 0:
            negated_bucket = "1" if int(n_negated_case_hits) == 1 else "2+"
            return f"negated_only__n{negated_bucket}"
        return "evidence_0"

    evidence_bucket = "1" if int(n_evidence_hits) == 1 else "2+"
    definitive_bucket = "0" if int(n_definitive_hits) == 0 else ("1" if int(n_definitive_hits) == 1 else "2+")
    uncertain_bucket = "0" if int(n_uncertain_hits) == 0 else ("1" if int(n_uncertain_hits) == 1 else "2+")

    if int(n_uncertain_hits) > 0 and int(n_definitive_hits) == 0:
        return f"uncertain_only__e{evidence_bucket}__u{uncertain_bucket}"

    if int(n_definitive_hits) > 0 and int(n_uncertain_hits) > 0:
        balance = (
            "uncertain_dominant"
            if int(n_uncertain_hits) >= int(n_definitive_hits)
            else "definite_dominant"
        )
        return (
            f"mixed__e{evidence_bucket}__d{definitive_bucket}"
            f"__u{uncertain_bucket}__{balance}"
        )

    if int(n_definitive_hits) > 0:
        return f"definite__e{evidence_bucket}__d{definitive_bucket}"

    return f"weak__e{evidence_bucket}"



def default_class_for_profile(profile_name: str) -> int:
    profile_name = str(profile_name)

    if profile_name == "evidence_0":
        return 0
    if profile_name.startswith("negated_only__"):
        return 0
    if profile_name.startswith("uncertain_only__"):
        return 2
    if profile_name.startswith("mixed__"):
        return 2 if "uncertain_dominant" in profile_name else 1
    if profile_name.startswith("definite__"):
        return 1
    if profile_name.startswith("weak__"):
        return 1

    return 1



def profile_is_uncertainty_leaning(profile_name: str) -> bool:
    profile_name = str(profile_name)
    return profile_name.startswith("uncertain_only__") or (
        profile_name.startswith("mixed__") and "uncertain_dominant" in profile_name
    )



def choose_positive_subclass(labels: pd.Series, profile_name: str) -> int:
    counts = labels.dropna().astype(int).value_counts().sort_index()
    positive_counts = counts.reindex(list(POSITIVE_CLASSES), fill_value=0).astype("float64")

    if positive_counts.sum() == 0:
        return default_class_for_profile(profile_name)

    weighted = positive_counts.copy()
    profile_name = str(profile_name)

    if profile_name.startswith("uncertain_only__"):
        weighted.loc[2] *= CLASS2_RECALL_BOOST * 1.30
    elif profile_name.startswith("mixed__"):
        if "uncertain_dominant" in profile_name:
            weighted.loc[2] *= CLASS2_RECALL_BOOST * 1.15
        else:
            weighted.loc[2] *= CLASS2_RECALL_BOOST * 0.90
            weighted.loc[1] *= CLASS1_DEFINITE_BOOST
    elif profile_name.startswith("definite__"):
        weighted.loc[1] *= CLASS1_DEFINITE_BOOST

    if profile_is_uncertainty_leaning(profile_name) and float(positive_counts.get(2, 0.0)) > 0.0:
        weighted_1 = float(weighted.get(1, 0.0))
        weighted_2 = float(weighted.get(2, 0.0))
        if weighted_2 >= max(1.0, weighted_1 * CLASS2_SELECTION_RATIO):
            return 2

    top_classes = weighted[weighted == weighted.max()].index.tolist()

    if len(top_classes) == 1:
        return int(top_classes[0])

    return int(default_class_for_profile(profile_name))



def learn_profile_to_class_recall_oriented(train_df: pd.DataFrame) -> tuple[dict[str, int], pd.DataFrame]:
    mapping: dict[str, int] = {}
    rows = []

    for profile_name, group in train_df.groupby("decision_profile"):
        labels = group["etiqueta_real"].dropna().astype(int)
        counts = labels.value_counts().sort_index()
        support = int(counts.sum())
        positive_support = int(counts.reindex(list(POSITIVE_CLASSES), fill_value=0).sum())
        positive_rate = (positive_support / support) if support else 0.0

        if profile_name == "evidence_0":
            chosen = 0
        elif positive_rate > MIN_POSITIVE_RATE_TO_KEEP_POSITIVE:
            chosen = choose_positive_subclass(labels, profile_name)
        else:
            chosen = 0

        mapping[str(profile_name)] = int(chosen)

        rows.append(
            {
                "decision_profile": str(profile_name),
                "support_train": support,
                "positive_support_train": positive_support,
                "positive_rate_train": round(float(positive_rate), 4),
                "assigned_class": int(chosen),
                "label_counts_train": dict(sorted(counts.to_dict().items())),
            }
        )

    diagnostics = pd.DataFrame(rows).sort_values(["decision_profile"]).reset_index(drop=True)
    return mapping, diagnostics



def stratified_train_test_indices(
    labels: pd.Series,
    train_fraction: float = 0.8,
    random_state: int = 42,
) -> tuple[list[int], list[int]]:
    valid_labels = pd.to_numeric(labels, errors="coerce").dropna().astype(int)

    if valid_labels.empty:
        return [], []

    train_indices: list[int] = []
    test_indices: list[int] = []

    for cls, cls_index in valid_labels.groupby(valid_labels).groups.items():
        cls_indices = list(cls_index)
        n_class = len(cls_indices)

        shuffled = pd.Series(cls_indices).sample(
            frac=1.0,
            random_state=int(random_state) + int(cls),
        ).tolist()

        if n_class == 1:
            n_train = 1
        else:
            n_train = int(round(n_class * float(train_fraction)))
            n_train = max(1, min(n_train, n_class - 1))

        train_indices.extend(shuffled[:n_train])
        test_indices.extend(shuffled[n_train:])

    if not test_indices and len(train_indices) > 1:
        shuffled_all = pd.Series(train_indices).sample(
            frac=1.0, random_state=int(random_state)
        ).tolist()
        moved_to_test = shuffled_all[-1]
        train_indices = [idx for idx in train_indices if idx != moved_to_test]
        test_indices = [moved_to_test]

    return sorted(train_indices), sorted(test_indices)



def balanced_mapping_without_labels(
    profiles: pd.Series,
    classes: tuple[int, ...] = (0, 1, 2),
) -> tuple[dict[str, int], dict[int, int]]:
    profile_counts = profiles.dropna().astype(str).value_counts().sort_values(ascending=False)

    if profile_counts.empty:
        return {}, {int(cls): 0 for cls in classes}

    mapping: dict[str, int] = {}
    class_totals = {int(cls): 0 for cls in classes}

    for profile_name, count in profile_counts.items():
        chosen = default_class_for_profile(str(profile_name))
        mapping[str(profile_name)] = int(chosen)
        class_totals[int(chosen)] += int(count)

    return mapping, class_totals



def metrics_for_class(
    y_true: pd.Series,
    y_pred: pd.Series,
    target_class: int,
) -> dict[str, float | int]:
    true_vals = pd.to_numeric(y_true, errors="coerce").astype("Int64")
    pred_vals = pd.to_numeric(y_pred, errors="coerce").astype("Int64")

    valid_mask = true_vals.notna() & pred_vals.notna()
    true_vals = true_vals[valid_mask].astype(int)
    pred_vals = pred_vals[valid_mask].astype(int)

    if true_vals.empty:
        return {
            "clase": int(target_class),
            "precision": 0.0,
            "sensibilidad": 0.0,
            "f1": 0.0,
            "soporte": 0,
        }

    tp = int(((pred_vals == target_class) & (true_vals == target_class)).sum())
    fp = int(((pred_vals == target_class) & (true_vals != target_class)).sum())
    fn = int(((pred_vals != target_class) & (true_vals == target_class)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    sensibilidad = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * sensibilidad / (precision + sensibilidad)) if (precision + sensibilidad) else 0.0

    return {
        "clase": int(target_class),
        "precision": float(precision),
        "sensibilidad": float(sensibilidad),
        "f1": float(f1),
        "soporte": int((true_vals == target_class).sum()),
    }



def class_metrics_for_targets(
    y_true: pd.Series,
    y_pred: pd.Series,
    target_classes: tuple[int, ...] = (1, 2),
) -> pd.DataFrame:
    rows = [metrics_for_class(y_true, y_pred, int(cls)) for cls in target_classes]
    metrics_df = pd.DataFrame(rows)

    if not metrics_df.empty:
        for col in ["precision", "sensibilidad", "f1"]:
            metrics_df[col] = metrics_df[col].round(4)

    return metrics_df



def any_positive_metrics(
    y_true: pd.Series,
    y_pred: pd.Series,
    positive_classes: tuple[int, ...] = POSITIVE_CLASSES,
) -> dict[str, float | int]:
    true_vals = pd.to_numeric(y_true, errors="coerce").astype("Int64")
    pred_vals = pd.to_numeric(y_pred, errors="coerce").astype("Int64")

    valid_mask = true_vals.notna() & pred_vals.notna()
    true_pos = true_vals[valid_mask].isin(list(positive_classes))
    pred_pos = pred_vals[valid_mask].isin(list(positive_classes))

    tp = int((pred_pos & true_pos).sum())
    fp = int((pred_pos & ~true_pos).sum())
    fn = int((~pred_pos & true_pos).sum())
    tn = int((~pred_pos & ~true_pos).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
        "specificity": round(float(specificity), 4),
    }



def clip_text(text: str, max_chars: int = 220) -> str:
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "..."





def compute_rule_scores(row: pd.Series) -> dict[int, float]:
    n_evidence = int(row.get("n_evidence_hits", 0) or 0)
    n_definitive = int(row.get("n_definitive_hits", 0) or 0)
    n_uncertain = int(row.get("n_uncertain_hits", 0) or 0)
    n_negated_case = int(row.get("n_negated_case_hits", 0) or 0)
    n_context_only = int(row.get("n_context_only_hits", 0) or 0)

    assign0 = bool(row.get("asignacion_directa_clase_0", False))
    assign2 = bool(row.get("asignacion_directa_clase_2", False))
    rule_state = str(row.get("rule_state", ""))
    decision_profile = str(row.get("decision_profile", ""))

    score_0 = 0.35
    score_1 = 0.35
    score_2 = 0.35

    # Evidencia a favor de clase 0
    score_0 += 1.85 * n_negated_case
    score_0 += 0.25 * n_context_only
    if n_evidence == 0:
        score_0 += 0.95
    if rule_state in {"negated_evidence", "no_evidence"}:
        score_0 += 0.25

    # Evidencia a favor de clase 1
    score_1 += 0.70 * n_evidence
    score_1 += 1.90 * n_definitive
    score_1 += 0.25 * max(n_evidence - n_uncertain, 0)
    if rule_state == "definite_positive":
        score_1 += 0.75
    if decision_profile.startswith("definite__"):
        score_1 += 0.50
    if decision_profile.startswith("mixed__") and "definite_dominant" in decision_profile:
        score_1 += 0.30

    # Evidencia a favor de clase 2
    score_2 += 0.70 * n_evidence
    score_2 += 1.95 * n_uncertain
    score_2 += 0.35 * max(n_uncertain - n_definitive, 0)
    if decision_profile.startswith("uncertain_only__"):
        score_2 += 0.70
    if decision_profile.startswith("mixed__") and "uncertain_dominant" in decision_profile:
        score_2 += 0.45

    # Penalizaciones cruzadas
    score_0 -= 0.80 * n_evidence
    score_0 -= 0.55 * n_definitive
    score_0 -= 0.15 * n_uncertain

    score_1 -= 0.80 * n_negated_case
    score_2 -= 0.45 * n_negated_case
    score_2 -= 0.40 * n_definitive

    # Refuerzo por asignaciones directas.
    # Esto conserva la lógica clínica fuerte del modelo de reglas.
    if assign0:
        score_0 += 4.50
        score_1 -= 1.50
        score_2 -= 1.25

    if assign2:
        score_2 += 4.50
        score_0 -= 1.50
        score_1 -= 1.00

    return {
        0: round(float(score_0), 6),
        1: round(float(score_1), 6),
        2: round(float(score_2), 6),
    }



def add_rule_scores(df: pd.DataFrame) -> pd.DataFrame:
    score_0 = []
    score_1 = []
    score_2 = []
    clase_sugerida_score = []
    score_top_1 = []
    score_top_2 = []
    margen_top_1_2 = []
    nivel_confianza_score = []
    score_clase_predicha = []
    coincide_score_vs_modelo = []

    for _, row in df.iterrows():
        scores = compute_rule_scores(row)
        ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)

        best_class, best_score = ordered[0]
        second_score = ordered[1][1] if len(ordered) > 1 else float("nan")
        margin = best_score - second_score if pd.notna(second_score) else float("nan")

        pred = row.get("clase_predicha", pd.NA)
        pred_class = int(pred) if pd.notna(pred) else None
        pred_score = scores.get(pred_class, float("nan")) if pred_class is not None else float("nan")

        if pd.isna(margin):
            confidence_level = pd.NA
        elif margin >= SCORE_MARGIN_HIGH:
            confidence_level = "alta"
        elif margin >= SCORE_MARGIN_MEDIUM:
            confidence_level = "media"
        else:
            confidence_level = "baja"

        score_0.append(scores[0])
        score_1.append(scores[1])
        score_2.append(scores[2])
        clase_sugerida_score.append(int(best_class))
        score_top_1.append(round(float(best_score), 6))
        score_top_2.append(round(float(second_score), 6) if pd.notna(second_score) else pd.NA)
        margen_top_1_2.append(round(float(margin), 6) if pd.notna(margin) else pd.NA)
        nivel_confianza_score.append(confidence_level)
        score_clase_predicha.append(round(float(pred_score), 6) if pd.notna(pred_score) else pd.NA)
        coincide_score_vs_modelo.append(
            bool(int(best_class) == pred_class) if pred_class is not None else pd.NA
        )

    df = df.copy()
    df["score_reglas_clase_0"] = score_0
    df["score_reglas_clase_1"] = score_1
    df["score_reglas_clase_2"] = score_2
    df["clase_sugerida_por_score"] = pd.Series(clase_sugerida_score, index=df.index, dtype="Int64")
    df["score_top_1"] = score_top_1
    df["score_top_2"] = score_top_2
    df["margen_score_top1_top2"] = margen_top_1_2
    df["nivel_confianza_score"] = nivel_confianza_score
    df["score_clase_predicha_reglas"] = score_clase_predicha
    df["coincide_score_vs_modelo"] = pd.Series(
        coincide_score_vs_modelo, index=df.index, dtype="boolean"
    )
    return df


## 5. Lectura De Datos


In [46]:

df = pd.read_excel(EXCEL_PATH, sheet_name=0, dtype=str, keep_default_na=False)

required_columns = {ID_COLUMN}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas obligatorias en el Excel: {missing}")

label_aliases = {
    normalize_for_match(LABEL_COLUMN),
    normalize_for_match("Mencion de cancer"),
}

resolved_label_column = None
for col in df.columns:
    if normalize_for_match(col) in label_aliases:
        resolved_label_column = col
        break

has_label_column = resolved_label_column is not None
excluded_columns = {ID_COLUMN}
if has_label_column:
    excluded_columns.add(resolved_label_column)

text_columns = [c for c in df.columns if c not in excluded_columns]
if not text_columns:
    raise ValueError("No se encontraron columnas de texto para procesar.")

if has_label_column:
    df["etiqueta_real"] = pd.to_numeric(df[resolved_label_column], errors="coerce").astype("Int64")
else:
    df["etiqueta_real"] = pd.Series([pd.NA] * len(df), dtype="Int64")

HAS_LABELS_FOR_EVAL = bool(df["etiqueta_real"].notna().any())

df["texto_modelo"] = df[text_columns].fillna("").apply(
    lambda row: row_to_text(row.values.tolist()), axis=1
)

print(f"Total de registros: {len(df)}")
if HAS_LABELS_FOR_EVAL:
    print(f"Registros etiquetados: {int(df['etiqueta_real'].notna().sum())}")
else:
    print("No se detectaron etiquetas validas.")


Total de registros: 4000
Registros etiquetados: 3999


## 6. Carga Del Modelo


In [47]:

print("\nCargando medSpaCy con QuickUMLS...")
nlp = medspacy.load(
    medspacy_enable=["medspacy_quickumls"],
    language_code="es",
)

if "cancer_keyword_component" not in nlp.pipe_names:
    nlp.add_pipe("cancer_keyword_component", last=True)



Cargando medSpaCy con QuickUMLS...
Loading QuickUMLS resources from a Medspacy-distributed SAMPLE of UMLS data from here: C:\Users\mafed\Documents\medspacy_nuevo\.venv\Lib\site-packages\resources\es\quickumls/QuickUMLS_SAMPLE_lowercase_Windows_unqlite


## 7. Extraccion De Reglas


In [48]:

print("Procesando textos...")
rule_states = []
decision_profiles_all = []
direct_class_0_all = []
direct_class_2_all = []
detected_terms_all = []
context_terms_all = []
valid_hits_all = []
ignored_hits_all = []
negated_hits_all = []
evidence_hits_all = []
negated_case_hits_all = []
context_only_hits_all = []
definitive_hits_all = []
uncertain_hits_all = []
family_hits_all = []
historical_hits_all = []

texts = df["texto_modelo"].fillna("").tolist()
total = max(len(texts), 1)

for i, doc in enumerate(nlp.pipe(texts, batch_size=32), start=1):
    features = extract_rule_features(doc)

    rule_states.append(features["rule_state"])
    decision_profiles_all.append(
        build_decision_profile(
            features["n_evidence_hits"],
            features["n_definitive_hits"],
            features["n_uncertain_hits"],
            features["n_negated_case_hits"],
        )
    )
    direct_class_0_all.append(features["assign_direct_class_0"])
    direct_class_2_all.append(features["assign_direct_class_2"])
    detected_terms_all.append("; ".join(features["detected_terms"]))
    context_terms_all.append("; ".join(features["context_terms"]))
    valid_hits_all.append(features["n_valid_hits"])
    ignored_hits_all.append(features["n_ignored_hits"])
    negated_hits_all.append(features["n_negated_hits"])
    evidence_hits_all.append(features["n_evidence_hits"])
    negated_case_hits_all.append(features["n_negated_case_hits"])
    context_only_hits_all.append(features["n_context_only_hits"])
    definitive_hits_all.append(features["n_definitive_hits"])
    uncertain_hits_all.append(features["n_uncertain_hits"])
    family_hits_all.append(features["n_family_hits"])
    historical_hits_all.append(features["n_historical_hits"])

    if i % 100 == 0 or i == total:
        pct = (i / total) * 100
        sys.stdout.write(f"\rProcesados: {i}/{total} ({pct:.1f}%)")
        sys.stdout.flush()

print()

df["rule_state"] = rule_states
df["decision_profile"] = decision_profiles_all
df["asignacion_directa_clase_0"] = pd.Series(direct_class_0_all, index=df.index, dtype="boolean")
df["asignacion_directa_clase_2"] = pd.Series(direct_class_2_all, index=df.index, dtype="boolean")
df["terminos_detectados"] = detected_terms_all
df["terminos_contexto_no_caso"] = context_terms_all
df["n_valid_hits"] = valid_hits_all
df["n_ignored_hits"] = ignored_hits_all
df["n_negated_hits"] = negated_hits_all
df["n_evidence_hits"] = evidence_hits_all
df["n_negated_case_hits"] = negated_case_hits_all
df["n_context_only_hits"] = context_only_hits_all
df["n_definitive_hits"] = definitive_hits_all
df["n_uncertain_hits"] = uncertain_hits_all
df["n_family_hits"] = family_hits_all
df["n_historical_hits"] = historical_hits_all


Procesando textos...
Procesados: 4000/4000 (100.0%)


## 8. Train / Test Y Mapeos


In [49]:

labeled_df = df[df["etiqueta_real"].notna()].copy()
train_df = pd.DataFrame()
validation_df = pd.DataFrame()
test_df = pd.DataFrame()
df["split"] = "unlabeled"
profile_diagnostics = pd.DataFrame()
profile_to_class: dict[str, int] = {}

if not labeled_df.empty:
    train_val_indices, test_indices = stratified_train_test_indices(
        labeled_df["etiqueta_real"],
        train_fraction=TRAIN_FRACTION,
        random_state=RANDOM_STATE,
    )

    train_val_df = df.loc[train_val_indices].copy()
    test_df = df.loc[test_indices].copy()

    train_indices, validation_indices = stratified_train_test_indices(
        train_val_df["etiqueta_real"],
        train_fraction=(1 - VALIDATION_WITHIN_TRAIN_FRACTION),
        random_state=RANDOM_STATE,
    )

    train_df = df.loc[train_indices].copy()
    validation_df = df.loc[validation_indices].copy()

    df.loc[train_df.index, "split"] = "train"
    df.loc[validation_df.index, "split"] = "validation"
    df.loc[test_df.index, "split"] = "test"

    direct_train_mask = (
        train_df["asignacion_directa_clase_2"].fillna(False)
        | train_df["asignacion_directa_clase_0"].fillna(False)
    )
    train_df_for_mapping = train_df[~direct_train_mask].copy()

    if train_df_for_mapping.empty:
        profile_to_class = {}
        profile_diagnostics = pd.DataFrame(columns=[
            "decision_profile", "support_train", "positive_support_train",
            "positive_rate_train", "assigned_class", "label_counts_train"
        ])
    else:
        profile_to_class, profile_diagnostics = learn_profile_to_class_recall_oriented(train_df_for_mapping)

    missing_profiles = sorted(
        set(
            df.loc[
                ~(
                    df["asignacion_directa_clase_2"].fillna(False)
                    | df["asignacion_directa_clase_0"].fillna(False)
                ), "decision_profile"
            ].dropna().astype(str).unique()
        ) - set(profile_to_class.keys())
    )
    for profile_name in missing_profiles:
        profile_to_class[profile_name] = default_class_for_profile(profile_name)
else:
    profile_to_class, _ = balanced_mapping_without_labels(df["decision_profile"])


## 9. Prediccion Final


In [50]:

df["clase_predicha"] = pd.Series([pd.NA] * len(df), index=df.index, dtype="Int64")

direct_class_0_mask = df["asignacion_directa_clase_0"].fillna(False)
df.loc[direct_class_0_mask, "clase_predicha"] = 0

direct_class_2_mask = df["asignacion_directa_clase_2"].fillna(False)
df.loc[direct_class_2_mask, "clase_predicha"] = 2

non_direct_mask = ~(direct_class_2_mask | direct_class_0_mask)
df.loc[non_direct_mask, "clase_predicha"] = (
    df.loc[non_direct_mask, "decision_profile"].map(profile_to_class)
)
df.loc[non_direct_mask, "clase_predicha"] = (
    df.loc[non_direct_mask, "clase_predicha"].fillna(
        df.loc[non_direct_mask, "decision_profile"].map(default_class_for_profile)
    )
)
df["clase_predicha"] = df["clase_predicha"].astype("Int64")
df = add_rule_scores(df)

resultado = df[[
    ID_COLUMN,
    "split",
    "clase_predicha",
    "clase_sugerida_por_score",
    "score_reglas_clase_0",
    "score_reglas_clase_1",
    "score_reglas_clase_2",
    "score_clase_predicha_reglas",
    "margen_score_top1_top2",
    "nivel_confianza_score",
    "coincide_score_vs_modelo",
]].copy()

if HAS_LABELS_FOR_EVAL:
    resultado["etiqueta_real"] = df["etiqueta_real"]
    resultado["acierto"] = (
        resultado["clase_predicha"].astype("string")
        == resultado["etiqueta_real"].astype("string")
    )


## 10. Metricas


In [51]:

metrics_all_train = pd.DataFrame()
metrics_12_train = pd.DataFrame()
confusion_train = pd.DataFrame()

metrics_all_validation = pd.DataFrame()
metrics_12_validation = pd.DataFrame()
confusion_validation = pd.DataFrame()

metrics_all = pd.DataFrame()
metrics_12 = pd.DataFrame()
metrics_summary_df = pd.DataFrame()
confusion = pd.DataFrame()

def evaluate_split(split_name: str):
    eval_df = df[(df["split"] == split_name) & df["etiqueta_real"].notna()].copy()

    if eval_df.empty:
        print(f"No hay registros en {split_name} para evaluar metricas.")
        return {
            "accuracy": None,
            "any_positive": None,
            "metrics_all": pd.DataFrame(),
            "metrics_12": pd.DataFrame(),
            "macro_f1_12": None,
            "macro_recall_12": None,
            "confusion": pd.DataFrame(),
        }

    print(f"\nEvaluacion sobre {split_name.upper()} (n={len(eval_df)} registros)")
    accuracy = float(
        (
            eval_df["clase_predicha"].astype("string")
            == eval_df["etiqueta_real"].astype("string")
        ).mean()
    )
    print(f"Accuracy en {split_name}: {accuracy:.4f}")

    any_positive = any_positive_metrics(
        eval_df["etiqueta_real"],
        eval_df["clase_predicha"],
        positive_classes=POSITIVE_CLASSES,
    )

    print("\nMetrica principal: deteccion de cualquier mencion de cancer (1 o 2) vs 0")
    for key in ["tp", "fp", "fn", "tn", "precision", "recall", "f1", "specificity"]:
        print(f"{key}: {any_positive[key]}")

    metrics_all_split = class_metrics_for_targets(
        eval_df["etiqueta_real"],
        eval_df["clase_predicha"],
        target_classes=(0, 1, 2),
    )
    print(f"\nMetricas por clase en {split_name}:")
    print(metrics_all_split.to_string(index=False))

    metrics_12_split = class_metrics_for_targets(
        eval_df["etiqueta_real"],
        eval_df["clase_predicha"],
        target_classes=PRIMARY_REVIEW_CLASSES,
    )
    macro_f1_12 = float(metrics_12_split["f1"].mean()) if not metrics_12_split.empty else 0.0
    macro_recall_12 = float(metrics_12_split["sensibilidad"].mean()) if not metrics_12_split.empty else 0.0
    print(f"\nMacro-F1 {split_name} clases 1 y 2: {macro_f1_12:.4f}")
    print(f"Macro-recall {split_name} clases 1 y 2: {macro_recall_12:.4f}")

    confusion_split = pd.crosstab(
        eval_df["etiqueta_real"].astype("Int64"),
        eval_df["clase_predicha"].astype("Int64"),
        rownames=["real"],
        colnames=["predicha"],
    ).reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0)

    print(f"\nMatriz de confusion completa ({split_name}):")
    print(confusion_split.to_string())

    real_2_row = confusion_split.loc[2]
    missed_2 = int(real_2_row[0] + real_2_row[1])
    print(f"\nResumen de la clase 2 real en {split_name}:")
    print(f"Detectados como 2: {int(real_2_row[2])}")
    print(f"No detectados como 2: {missed_2}")
    print(f"De los no detectados como 2, quedaron como 0: {int(real_2_row[0])}")
    print(f"De los no detectados como 2, quedaron como 1: {int(real_2_row[1])}")

    return {
        "accuracy": accuracy,
        "any_positive": any_positive,
        "metrics_all": metrics_all_split,
        "metrics_12": metrics_12_split,
        "macro_f1_12": macro_f1_12,
        "macro_recall_12": macro_recall_12,
        "confusion": confusion_split,
    }

if HAS_LABELS_FOR_EVAL:
    train_results = evaluate_split("train")
    validation_results = evaluate_split("validation")
    test_results = evaluate_split("test")

    metrics_all_train = train_results["metrics_all"]
    metrics_12_train = train_results["metrics_12"]
    confusion_train = train_results["confusion"]

    metrics_all_validation = validation_results["metrics_all"]
    metrics_12_validation = validation_results["metrics_12"]
    confusion_validation = validation_results["confusion"]

    metrics_all = test_results["metrics_all"]
    metrics_12 = test_results["metrics_12"]
    confusion = test_results["confusion"]

    summary_rows = []
    for split_name, split_results in [
        ("train", train_results),
        ("validation", validation_results),
        ("test", test_results),
    ]:
        if split_results["accuracy"] is None:
            continue
        any_positive = split_results["any_positive"]
        summary_rows.extend([
            {"split": split_name, "metrica": f"accuracy_{split_name}", "valor": round(float(split_results["accuracy"]), 4)},
            {"split": split_name, "metrica": "tp_any_positive", "valor": any_positive["tp"]},
            {"split": split_name, "metrica": "fp_any_positive", "valor": any_positive["fp"]},
            {"split": split_name, "metrica": "fn_any_positive", "valor": any_positive["fn"]},
            {"split": split_name, "metrica": "tn_any_positive", "valor": any_positive["tn"]},
            {"split": split_name, "metrica": "precision_any_positive", "valor": any_positive["precision"]},
            {"split": split_name, "metrica": "recall_any_positive", "valor": any_positive["recall"]},
            {"split": split_name, "metrica": "f1_any_positive", "valor": any_positive["f1"]},
            {"split": split_name, "metrica": "specificity_any_positive", "valor": any_positive["specificity"]},
            {"split": split_name, "metrica": "macro_f1_clases_1_2", "valor": round(float(split_results["macro_f1_12"]), 4)},
            {"split": split_name, "metrica": "macro_recall_clases_1_2", "valor": round(float(split_results["macro_recall_12"]), 4)},
        ])
    metrics_summary_df = pd.DataFrame(summary_rows)

final_class_counts = (
    df["clase_predicha"]
    .dropna()
    .astype(int)
    .value_counts()
    .reindex([0, 1, 2], fill_value=0)
    .sort_index()
)

print("\nConteo final de registros por clase predicha:")
print(final_class_counts.rename_axis("clase").to_frame("n").to_string())



Evaluacion sobre TRAIN (n=2560 registros)
Accuracy en train: 0.9559

Metrica principal: deteccion de cualquier mencion de cancer (1 o 2) vs 0
tp: 568
fp: 57
fn: 38
tn: 1897
precision: 0.9088
recall: 0.9373
f1: 0.9228
specificity: 0.9708

Metricas por clase en train:
 clase  precision  sensibilidad     f1  soporte
     0     0.9804        0.9708 0.9756     1954
     1     0.8883        0.9312 0.9092      581
     2     0.5625        0.3600 0.4390       25

Macro-F1 train clases 1 y 2: 0.6741
Macro-recall train clases 1 y 2: 0.6456

Matriz de confusion completa (train):
predicha     0    1  2
real                  
0         1897   57  0
1           33  541  7
2            5   11  9

Resumen de la clase 2 real en train:
Detectados como 2: 9
No detectados como 2: 16
De los no detectados como 2, quedaron como 0: 5
De los no detectados como 2, quedaron como 1: 11

Evaluacion sobre VALIDATION (n=639 registros)
Accuracy en validation: 0.9609

Metrica principal: deteccion de cualquier mencion

## 11. Exportar Excel


In [52]:

output_path = OUTPUT_PATH
if output_path.exists():
    try:
        output_path.unlink()
    except PermissionError:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = output_path.with_name(f"{output_path.stem}_{timestamp}{output_path.suffix}")

salida_simple = df[[ID_COLUMN, "clase_predicha"]].copy()
salida_simple = salida_simple.rename(columns={ID_COLUMN: "id"})

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    salida_simple.to_excel(writer, sheet_name="salida_simple", index=False)

print(f"\nArchivo guardado en:\n{output_path}")


Archivo guardado en:
C:\Users\mafed\Desktop\Trabajo de grado\Datos\Resultados\Reglas\base etiquetada limpia_clasificada_reglas_quickumls_con_scores.xlsx


## 12. Información sobre los que fallan

In [53]:


resumen_final = df[[ID_COLUMN, "clase_predicha"]].copy()
resumen_final = resumen_final.rename(columns={
    ID_COLUMN: "id",
    "clase_predicha": "y_pred",
})

if "etiqueta_real" in df.columns:
    resumen_final.insert(1, "y_true", df["etiqueta_real"].values)

if "score_reglas_clase_0" in df.columns:
    resumen_final["score_0"] = df["score_reglas_clase_0"].values

if "score_reglas_clase_1" in df.columns:
    resumen_final["score_1"] = df["score_reglas_clase_1"].values

if "score_reglas_clase_2" in df.columns:
    resumen_final["score_2"] = df["score_reglas_clase_2"].values

if "nivel_confianza_score" in df.columns:
    resumen_final["confianza"] = df["nivel_confianza_score"].values

orden_final = ["id"]
if "y_true" in resumen_final.columns:
    orden_final.append("y_true")
orden_final.append("y_pred")

for c in ["score_0", "score_1", "score_2", "confianza"]:
    if c in resumen_final.columns:
        orden_final.append(c)

resumen_final = resumen_final[orden_final]

print("Vista previa del resumen final:")
display(resumen_final.head())

output_resumen = Path("resumen_final_reglas.xlsx")
if output_resumen.exists():
    try:
        output_resumen.unlink()
    except PermissionError:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_resumen = output_resumen.with_name(f"{output_resumen.stem}_{timestamp}{output_resumen.suffix}")

resumen_final.to_excel(output_resumen, index=False)
print(f"\nArchivo guardado en: {output_resumen.resolve()}")

Vista previa del resumen final:


,id,y_true,y_pred,score_0,score_1,score_2,confianza
0,CASO_000001,0,0,1.55,0.35,0.35,media
1,CASO_000002,0,0,1.55,0.35,0.35,media
2,CASO_000003,0,0,1.55,0.35,0.35,media
3,CASO_000004,0,0,1.55,0.35,0.35,media
4,CASO_000005,0,0,1.55,0.35,0.35,media



Archivo guardado en: C:\Users\mafed\Documents\medspacy_nuevo\resumen_final_reglas.xlsx
